In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

# IR-CoT

## 阶段一：将问题分解为子问题

In [2]:
from langchain_core.prompts import ChatPromptTemplate

# 问题分解
template = """你是一个能根据输入问题生成多个子问题的有用助手。
目标是将输入分解为一组可以独立回答的子问题。
生成与原始问题相关的 3 个子问题。
原始问题：{question}。
输出 3 个可以独立回答的子问题："""

prompt_decomposition = ChatPromptTemplate.from_messages(
    [
        ("system", template)
    ]
)

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# LLM
llm = ChatOpenAI(temperature=0)

# Output Parser
output_parser = StrOutputParser()

# Chain
generate_querires_decomposition = prompt_decomposition | llm | output_parser | (lambda x: x.strip().split("\n"))

In [4]:
question = "zero shot 与 few shot 的区别是什么？"

In [5]:
# Run
sub_questions = generate_querires_decomposition.invoke({"question": question})

sub_questions

['1. zero shot 学习方法的原理是什么？',
 '2. few shot 学习方法如何应用于机器学习领域？',
 '3. zero shot 和 few shot 学习方法在实际应用中有哪些区别和优劣势？']

## 阶段二：依次解决子问题

### 索引

In [6]:
from dotenv import load_dotenv
from langchain import hub
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载环境变量
load_dotenv()

# Load
file_path = "./data/translated_article.txt"
loader = TextLoader(file_path, encoding='utf-8')
docs = loader.load()

# Split
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)
markdown_text = "\n\n".join([doc.page_content for doc in docs])

md_header_splits = markdown_splitter.split_text(markdown_text)

chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(md_header_splits)

# Embed
embedding = OpenAIEmbeddings()

# Store
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

# Retrieve
retriever = vectorstore.as_retriever()

### 检索与生成

In [7]:
# Prompt
template = """以下是您需要回答的问题：

\n --- \n {question} \n --- \n

以下是所有可用的背景问题和答案对：

\n --- \n {q_a_pairs} \n --- \n

以下是与问题相关的附加背景信息：

\n --- \n {context} \n --- \n

使用以上背景信息和任何背景问答对来回答问题：\n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", template)
    ]
)

In [9]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

# Chain
def format_qa_pair(question, answer):
    """格式化问答对"""
    
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

q_a_pairs = ""
for sub_question in sub_questions:
    rag_chain = (
    {"context": itemgetter("question") | retriever | format_docs, 
     "question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs")} 
    | decomposition_prompt
    | llm
    | StrOutputParser())

    sub_answer = rag_chain.invoke({"question":sub_question, "q_a_pairs":q_a_pairs})
    q_a_pair = format_qa_pair(sub_question, sub_answer)
    q_a_pairs = q_a_pairs + "\n---\n"+ q_a_pair

Tips:

itemgetter 是 Python 的 operator 模块中的一个工具，主要用于从对象中获取特定的项。

`itemgetter("question")`，该函数从给定对象中提取键为 "question" 的值

即从给定对象（{"question":sub_question, "q_a_pairs":q_a_pairs}）中提取键为 "question" 的值（sub_question）

In [10]:
print(q_a_pairs)


---
Question: 1. zero shot 学习方法的原理是什么？
Answer: 零样本学习方法的原理是在没有任何示例或标签的情况下，直接将任务文本输入模型并询问结果。这种方法依赖于模型在训练阶段学习到的语言表示和知识，以及对任务的理解能力。在零样本学习中，模型需要通过推理和泛化能力来预测正确的结果，而不是简单地记忆训练数据中的示例。通过合理设计提示格式和选择合适的训练示例，可以帮助模型在零样本学习任务中取得更好的性能。
---
Question: 2. few shot 学习方法如何应用于机器学习领域？
Answer: Few-shot learning方法在机器学习领域中被广泛应用，特别是在自然语言处理和计算机视觉任务中。这种方法通过提供少量的训练示例来训练模型，使其能够在面对新任务或类别时进行有效的泛化。在自然语言处理领域，Few-shot learning方法可以帮助模型在只有少量标记数据的情况下进行文本分类、命名实体识别、情感分析等任务。在计算机视觉领域，Few-shot learning方法可以用于目标检测、图像分类、图像生成等任务，通过提供少量的示例来训练模型以适应新的类别或场景。通过Few-shot learning方法，模型可以更好地适应新任务，减少对大量标记数据的依赖，提高模型的泛化能力和效率。
---
Question: 3. zero shot 和 few shot 学习方法在实际应用中有哪些区别和优劣势？
Answer: 零样本（Zero-shot）和少样本（Few-shot）学习方法在实际应用中有以下区别和优劣势：

### 区别：
1. **零样本学习**：
   - **原理**：零样本学习方法在没有任何示例或标签的情况下，直接将任务文本输入模型并询问结果。
   - **特点**：模型需要通过推理和泛化能力来预测正确的结果，依赖于模型在训练阶段学习到的语言表示和知识。
   - **应用**：适用于完全新的任务或类别，可以帮助模型在没有任何训练数据的情况下进行预测。

2. **少样本学习**：
   - **原理**：少样本学习通过提供少量高质量的示例来训练模型，使其能够在面对新任务或类别时进行有效的泛化。
   - **特点**：相对于零样本学习，少样本学习通常表现更好，因为提供了一些示例来帮助模型理解任务和评价标

In [11]:
question

'zero shot 与 few shot 的区别是什么？'

In [12]:
print(sub_answer)

零样本（Zero-shot）和少样本（Few-shot）学习方法在实际应用中有以下区别和优劣势：

### 区别：
1. **零样本学习**：
   - **原理**：零样本学习方法在没有任何示例或标签的情况下，直接将任务文本输入模型并询问结果。
   - **特点**：模型需要通过推理和泛化能力来预测正确的结果，依赖于模型在训练阶段学习到的语言表示和知识。
   - **应用**：适用于完全新的任务或类别，可以帮助模型在没有任何训练数据的情况下进行预测。

2. **少样本学习**：
   - **原理**：少样本学习通过提供少量高质量的示例来训练模型，使其能够在面对新任务或类别时进行有效的泛化。
   - **特点**：相对于零样本学习，少样本学习通常表现更好，因为提供了一些示例来帮助模型理解任务和评价标准。
   - **应用**：适用于需要在有限标记数据情况下进行训练的任务，可以帮助模型更好地适应新任务或类别。

### 优劣势：
1. **零样本学习**：
   - **优势**：
     - 适用于完全新的任务或类别，无需提供任何示例。
     - 可以帮助模型在没有训练数据的情况下进行预测。
   - **劣势**：
     - 对模型的泛化能力和推理能力要求较高，性能可能不如少样本学习。
     - 在某些复杂任务上可能表现不佳，因为缺乏示例来帮助模型理解任务。

2. **少样本学习**：
   - **优势**：
     - 通常比零样本学习表现更好，因为提供了一些示例来帮助模型理解任务。
     - 可以在有限标记数据情况下训练模型，提高泛化能力。
   - **劣势**：
     - 需要更多的训练示例，可能会增加数据收集和标记的成本。
     - 当示例数量有限时，模型可能仍然面临泛化能力不足的挑战。

综上所述，零样本学习适用于完全新的任务，而少样本学习适用于有限标记数据情况下的任务。选择合适的学习方法取决于任务的特点和可用数据量。
